# 01 - Analytic Pricing and Greeks

Black-Scholes-Merton pricing with the full Greek set, checked against finite differences,
implied volatility by Newton-Raphson, and the binomial and trinomial lattices converging
to the closed form. Each result is validated against an independent calculation so a
mistake shows up as a disagreement.


## Setup

In [ ]:
!pip install -q numpy scipy matplotlib pandas

import subprocess, os, sys
from google.colab import userdata

GITHUB_USER = "your-username"   # <-- edit this
REPO = "derivatives-pricing"

gh_token = userdata.get("GITHUB_TOKEN")
url = f"https://{gh_token}@github.com/{GITHUB_USER}/{REPO}.git"
r = subprocess.run(["git", "clone", url], capture_output=True, text=True)
if r.returncode == 0 or "already exists" in r.stderr:
    os.chdir(REPO)
subprocess.run(["git", "pull", "--rebase", "origin", "main"], capture_output=True, text=True)

!pip install -q -e .
src = os.path.abspath("src")
if src not in sys.path:
    sys.path.insert(0, src)
import derivpricing
print("derivpricing", derivpricing.__version__)

## Black-Scholes price and Greeks

The price and every Greek come from one call. Vega and rho are quoted per unit change in
volatility and rate; theta is per year.


In [ ]:
from derivpricing.analytic import black_scholes, bs_price

S0, K, T, r, sigma = 100.0, 100.0, 1.0, 0.05, 0.2
call = black_scholes(S0, K, T, r, sigma, "call")
put = black_scholes(S0, K, T, r, sigma, "put")
print(f"Call {call['price']:.4f}   Put {put['price']:.4f}")
for g in ["delta", "gamma", "vega", "theta", "rho"]:
    print(f"  call {g:6s} {call[g]:+.4f}")

## Greeks against finite differences

A closed-form Greek and a central finite difference of the price should agree. Matching
them is the check that the analytic derivatives are right.


In [ ]:
import pandas as pd
h = 1e-4
fd = {
    "delta": (bs_price(S0+h,K,T,r,sigma) - bs_price(S0-h,K,T,r,sigma)) / (2*h),
    "gamma": (bs_price(S0+h,K,T,r,sigma) - 2*bs_price(S0,K,T,r,sigma) + bs_price(S0-h,K,T,r,sigma)) / h**2,
    "vega":  (bs_price(S0,K,T,r,sigma+h) - bs_price(S0,K,T,r,sigma-h)) / (2*h),
    "rho":   (bs_price(S0,K,T,r+h,sigma) - bs_price(S0,K,T,r-h,sigma)) / (2*h),
}
rows = [{"greek": g, "analytic": round(call[g],6), "finite_diff": round(fd[g],6),
         "abs_diff": abs(call[g]-fd[g])} for g in fd]
pd.DataFrame(rows).set_index("greek")

## Implied volatility round-trip

Feeding a Black-Scholes price back through the Newton-Raphson solver should return the
volatility that produced it.


In [ ]:
from derivpricing.analytic import implied_vol
price = bs_price(S0, K, T, r, sigma, "call")
iv, iters = implied_vol(price, S0, K, T, r, "call")
print(f"input sigma {sigma}  ->  implied vol {iv:.6f}  in {iters} iterations")

## Lattice convergence

The CRR binomial and Boyle trinomial prices approach the Black-Scholes value as the step
count grows. Plotting the price against $N$ shows both settling onto the analytic line.


In [ ]:
import numpy as np, matplotlib.pyplot as plt
from derivpricing.trees import binomial_tree, trinomial_tree

Ns = np.arange(10, 400, 10)
bs = bs_price(S0, K, T, r, sigma, "call")
bino = [binomial_tree(S0, K, T, r, sigma, int(n), "call") for n in Ns]
trino = [trinomial_tree(S0, K, T, r, sigma, int(n), "call") for n in Ns]

plt.figure(figsize=(8, 4.5))
plt.plot(Ns, bino, label="binomial", alpha=0.8)
plt.plot(Ns, trino, label="trinomial", alpha=0.8)
plt.axhline(bs, color="black", ls="--", label=f"Black-Scholes {bs:.4f}")
plt.xlabel("steps N"); plt.ylabel("call price"); plt.legend(); plt.title("Lattice convergence to Black-Scholes")
plt.tight_layout(); plt.show()

## Recap

The analytic prices, the finite-difference-checked Greeks, the implied-vol round-trip, and
the converging lattices give four mutually consistent views of the same vanilla option.
Notebook `02` moves to Monte Carlo and its sampling error.
